<a href="https://colab.research.google.com/github/AshokGit544/enterprise-finance-master-data-resolution-platform./blob/main/enterprise_finance_master_data_resolution_platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import warnings
warnings.filterwarnings("ignore")

import random
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
np.random.seed(42)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 50)

PROJECT_DIR = Path("enterprise_finance_master_data_resolution_output")
PROJECT_DIR.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())

def make_base_vendor_names():
    return [
        "Alpha Grid Services",
        "Northwind Electric Supply",
        "GreenVolt Systems",
        "Prime Utility Parts",
        "TransEnergy Logistics",
        "Vertex Maintenance",
        "BluePeak Consulting",
        "SolarEdge Components",
        "Metro Industrial Tools",
        "Crescent Field Ops",
        "Pioneer Cable Works",
        "Helios Analytics",
        "Summit Procurement",
        "IronBridge Equipment",
        "EnerSys Contractors",
        "Utility Edge Networks",
        "GridPoint Services",
        "VoltBridge Technologies",
        "EverField Maintenance",
        "PowerLane Utility Works"
    ]

def make_vendor_variants(name):
    parts = [
        name,
        name + " LLC",
        name + " Inc",
        name + " Corporation",
        name.replace("Services", "Svc"),
        name.replace("Systems", "Sys"),
        name.replace("Technologies", "Tech"),
        name.replace("Maintenance", "Maint"),
        name.replace("Consulting", "Consult"),
        name.replace("Utility", "Util"),
        name.upper(),
        name.lower(),
    ]
    return list(dict.fromkeys(parts))

def make_vendor_master(system_name, n_records=140):
    base_names = make_base_vendor_names()
    categories = ["Maintenance", "IT Services", "Electrical Components", "Logistics", "Consulting", "Field Services"]
    states = ["NY", "CT", "ME", "MA", "PA", "NJ", "RI"]
    rows = []

    for i in range(1, n_records + 1):
        canonical = random.choice(base_names)
        variant = random.choice(make_vendor_variants(canonical))

        tax_base = abs(hash(canonical)) % 1000000000
        tax_id = str(100000000 + tax_base % 900000000)

        if random.random() < 0.18:
            tax_id = tax_id[:-1] + str(random.randint(0, 9))

        address_num = random.randint(100, 999)
        address_street = random.choice(["Main St", "Market St", "Broadway", "Industrial Rd", "River Ave", "Lake Dr"])
        city = random.choice(["White Plains", "Hartford", "Newark", "Boston", "Albany", "Providence", "Portland"])

        if random.random() < 0.15:
            variant = variant.replace(" ", "  ")
        if random.random() < 0.10:
            variant = variant.replace("and", "&")

        rows.append({
            "source_system": system_name,
            "source_vendor_id": f"{system_name[:3].upper()}_{i:05d}",
            "vendor_name_raw": variant,
            "tax_id": tax_id,
            "vendor_category": random.choice(categories),
            "vendor_state": random.choice(states),
            "address_line_1": f"{address_num} {address_street}",
            "city": city,
            "payment_terms_days": random.choice([15, 30, 45, 60]),
            "risk_rating": random.choice(["Low", "Medium", "High"]),
            "active_flag": random.choice([1, 1, 1, 0]),
            "canonical_seed_name": canonical
        })

    return pd.DataFrame(rows)

sap_fi_vendors = make_vendor_master("SAP_FI", 160)
sap_ap_vendors = make_vendor_master("SAP_AP", 150)
erp_stage_vendors = make_vendor_master("ERP_STAGE", 150)

raw_vendor_master = pd.concat([sap_fi_vendors, sap_ap_vendors, erp_stage_vendors], ignore_index=True)
raw_vendor_master.to_csv(PROJECT_DIR / "raw_vendor_master.csv", index=False)

print("Raw vendor master shape:", raw_vendor_master.shape)
display(raw_vendor_master.head())

def normalize_vendor_name(name):
    if pd.isna(name):
        return ""
    cleaned = str(name).upper().strip()
    replacements = {
        " LLC": "",
        " INC": "",
        " CORPORATION": "",
        " CORP": "",
        " CO": "",
        " SVC": " SERVICES",
        " SYS": " SYSTEMS",
        " TECH": " TECHNOLOGIES",
        " MAINT": " MAINTENANCE",
        " CONSULT": " CONSULTING",
        " UTIL": " UTILITY",
        "  ": " "
    }
    for old, new in replacements.items():
        cleaned = cleaned.replace(old, new)
    return " ".join(cleaned.split())

vendor_master = raw_vendor_master.copy()
vendor_master["vendor_name_normalized"] = vendor_master["vendor_name_raw"].apply(normalize_vendor_name)
vendor_master["address_normalized"] = vendor_master["address_line_1"].str.upper().str.strip()
vendor_master["city_normalized"] = vendor_master["city"].str.upper().str.strip()

vendor_master.to_csv(PROJECT_DIR / "vendor_master_normalized.csv", index=False)

vendor_master["match_document"] = vendor_master.apply(
    lambda r: (
        f"vendor name {r['vendor_name_normalized']} "
        f"tax id {r['tax_id']} "
        f"category {r['vendor_category']} "
        f"state {r['vendor_state']} "
        f"address {r['address_normalized']} "
        f"city {r['city_normalized']}"
    ),
    axis=1
)

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
X = vectorizer.fit_transform(vendor_master["match_document"])

sim_matrix = cosine_similarity(X)

threshold = 0.72
n = len(vendor_master)
parent = list(range(n))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[ry] = rx

for i in range(n):
    for j in range(i + 1, n):
        score = sim_matrix[i, j]
        same_tax = vendor_master.iloc[i]["tax_id"] == vendor_master.iloc[j]["tax_id"]
        same_name = vendor_master.iloc[i]["vendor_name_normalized"] == vendor_master.iloc[j]["vendor_name_normalized"]

        if (score >= threshold and same_tax) or (same_name and score >= 0.55):
            union(i, j)

cluster_ids = [find(i) for i in range(n)]
vendor_master["match_cluster_id"] = pd.factorize(cluster_ids)[0] + 1
vendor_master["golden_vendor_id"] = vendor_master["match_cluster_id"].apply(lambda x: f"GV{x:05d}")

match_pairs = []
for i in range(n):
    for j in range(i + 1, n):
        if vendor_master.iloc[i]["golden_vendor_id"] == vendor_master.iloc[j]["golden_vendor_id"]:
            match_pairs.append({
                "record_1_system": vendor_master.iloc[i]["source_system"],
                "record_1_vendor_id": vendor_master.iloc[i]["source_vendor_id"],
                "record_1_name": vendor_master.iloc[i]["vendor_name_raw"],
                "record_2_system": vendor_master.iloc[j]["source_system"],
                "record_2_vendor_id": vendor_master.iloc[j]["source_vendor_id"],
                "record_2_name": vendor_master.iloc[j]["vendor_name_raw"],
                "similarity_score": round(float(sim_matrix[i, j]), 4),
                "golden_vendor_id": vendor_master.iloc[i]["golden_vendor_id"]
            })

match_pairs_df = pd.DataFrame(match_pairs)
match_pairs_df.to_csv(PROJECT_DIR / "vendor_match_pairs.csv", index=False)

def build_golden_record(group):
    best_name = group["vendor_name_normalized"].mode().iloc[0]
    best_tax_id = group["tax_id"].mode().iloc[0]
    preferred_category = group["vendor_category"].mode().iloc[0]
    preferred_state = group["vendor_state"].mode().iloc[0]
    common_terms = group["payment_terms_days"].mode().iloc[0]
    risk = group["risk_rating"].mode().iloc[0]
    active = int(group["active_flag"].max())

    return pd.Series({
        "golden_vendor_name": best_name,
        "golden_tax_id": best_tax_id,
        "vendor_category": preferred_category,
        "vendor_state": preferred_state,
        "payment_terms_days": common_terms,
        "risk_rating": risk,
        "active_flag": active,
        "source_record_count": len(group),
        "source_systems": ", ".join(sorted(group["source_system"].unique()))
    })

golden_vendor_master = (
    vendor_master.groupby("golden_vendor_id")
    .apply(build_golden_record)
    .reset_index()
)

golden_vendor_master.to_csv(PROJECT_DIR / "golden_vendor_master.csv", index=False)

print("Golden vendor master shape:", golden_vendor_master.shape)
display(golden_vendor_master.head())

def make_cost_centers():
    rows = [
        ("CC100", "Transmission Operations", "Operations"),
        ("CC110", "Distribution Maintenance", "Operations"),
        ("CC120", "Substation Engineering", "Engineering"),
        ("CC130", "Renewables Program", "Strategy"),
        ("CC140", "Customer Support", "Customer Ops"),
        ("CC150", "IT Shared Services", "Technology"),
        ("CC160", "Finance Operations", "Finance"),
        ("CC170", "Vegetation Management", "Field Ops"),
        ("CC180", "Grid Modernization", "Engineering"),
        ("CC190", "Regulatory Affairs", "Corporate")
    ]
    return pd.DataFrame(rows, columns=["cost_center_id", "cost_center_name", "department"])

def make_profit_centers():
    rows = [
        ("PC200", "Electric Distribution East", "East"),
        ("PC210", "Electric Distribution West", "West"),
        ("PC220", "Gas and Field Services", "North"),
        ("PC230", "Renewables and Storage", "Enterprise"),
        ("PC240", "Corporate Shared Services", "Corporate")
    ]
    return pd.DataFrame(rows, columns=["profit_center_id", "profit_center_name", "region_group"])

def make_gl_accounts():
    rows = [
        ("400100", "Revenue - Energy Delivery", "Revenue"),
        ("500100", "Opex - Maintenance", "Expense"),
        ("500200", "Opex - IT Services", "Expense"),
        ("500300", "Opex - Contractors", "Expense"),
        ("500400", "Opex - Logistics", "Expense"),
        ("500500", "Opex - Vegetation Mgmt", "Expense"),
        ("500600", "Opex - Consulting", "Expense"),
        ("500700", "Opex - Emergency Repairs", "Expense"),
        ("500800", "Capex - Grid Modernization", "Capital"),
        ("500900", "Capex - Substation Upgrades", "Capital"),
        ("110100", "Accounts Payable", "Liability")
    ]
    return pd.DataFrame(rows, columns=["gl_account", "gl_account_name", "gl_type"])

cost_centers = make_cost_centers()
profit_centers = make_profit_centers()
gl_accounts = make_gl_accounts()

cost_centers.to_csv(PROJECT_DIR / "cost_centers.csv", index=False)
profit_centers.to_csv(PROJECT_DIR / "profit_centers.csv", index=False)
gl_accounts.to_csv(PROJECT_DIR / "gl_accounts.csv", index=False)

descriptions = [
    "substation maintenance support",
    "vegetation clearing around power lines",
    "grid inspection contractor services",
    "cybersecurity monitoring support",
    "cloud cost optimization consulting",
    "field equipment transportation",
    "metering systems enhancement",
    "SCADA support services",
    "storm response contractor mobilization",
    "distribution pole inspection",
    "asset reliability assessment",
    "emergency transformer replacement",
    "temporary field crew support",
    "circuit restoration activities",
    "renewables integration support",
    "substation modernization project"
]

def generate_finance_transactions(vendor_source_df, num_rows=7000):
    start_date = datetime(2024, 1, 1)
    end_date = datetime(2025, 12, 31)
    date_range_days = (end_date - start_date).days
    rows = []

    for i in range(1, num_rows + 1):
        v = vendor_source_df.sample(1).iloc[0]
        cc = cost_centers.sample(1).iloc[0]
        pc = profit_centers.sample(1).iloc[0]
        gl = gl_accounts.sample(1).iloc[0]

        posting_date = start_date + timedelta(days=random.randint(0, date_range_days))
        due_date = posting_date + timedelta(days=int(v["payment_terms_days"]))
        payment_date = due_date + timedelta(days=random.randint(-5, 30))

        amount = random.uniform(3000, 120000)
        if v["vendor_category"] == "Consulting":
            amount *= 1.15
        if gl["gl_type"] == "Capital":
            amount *= 1.22
        if cc["cost_center_name"] == "Vegetation Management":
            amount *= 1.10

        rows.append({
            "document_id": f"DOC{i:06d}",
            "invoice_id": f"INV{i:06d}",
            "source_system": v["source_system"],
            "source_vendor_id": v["source_vendor_id"],
            "posting_date": posting_date.date().isoformat(),
            "due_date": due_date.date().isoformat(),
            "payment_date": payment_date.date().isoformat(),
            "cost_center_id": cc["cost_center_id"],
            "profit_center_id": pc["profit_center_id"],
            "gl_account": gl["gl_account"],
            "amount_usd": round(amount, 2),
            "currency": "USD",
            "description": random.choice(descriptions),
            "status": random.choice(["POSTED", "APPROVED", "PAID", "HOLD"])
        })

    return pd.DataFrame(rows)

finance_transactions = generate_finance_transactions(vendor_master)
finance_transactions.to_csv(PROJECT_DIR / "finance_transactions_raw.csv", index=False)

transactions_enriched = (
    finance_transactions
    .merge(vendor_master[[
        "source_system", "source_vendor_id", "golden_vendor_id", "vendor_name_raw",
        "vendor_name_normalized", "vendor_category", "vendor_state", "risk_rating"
    ]], on=["source_system", "source_vendor_id"], how="left")
    .merge(golden_vendor_master, on="golden_vendor_id", how="left")
    .merge(cost_centers, on="cost_center_id", how="left")
    .merge(profit_centers, on="profit_center_id", how="left")
    .merge(gl_accounts, on="gl_account", how="left")
)

transactions_enriched["posting_date"] = pd.to_datetime(transactions_enriched["posting_date"])
transactions_enriched["posting_month"] = transactions_enriched["posting_date"].dt.to_period("M").astype(str)
transactions_enriched["duplicate_vendor_record_flag"] = (
    transactions_enriched.groupby("golden_vendor_id")["source_vendor_id"].transform("nunique") > 1
).astype(int)

transactions_enriched.to_csv(PROJECT_DIR / "finance_transactions_resolved.csv", index=False)

vendor_360 = (
    transactions_enriched.groupby(
        ["golden_vendor_id", "golden_vendor_name", "vendor_category", "vendor_state", "risk_rating", "source_record_count"],
        as_index=False
    )
    .agg(
        total_spend_usd=("amount_usd", "sum"),
        transaction_count=("document_id", "count"),
        distinct_source_vendor_ids=("source_vendor_id", "nunique"),
        distinct_cost_centers=("cost_center_id", "nunique"),
        distinct_gl_accounts=("gl_account", "nunique")
    )
    .sort_values("total_spend_usd", ascending=False)
)

vendor_360["avg_transaction_amount"] = (
    vendor_360["total_spend_usd"] / vendor_360["transaction_count"]
).round(2)

vendor_360.to_csv(PROJECT_DIR / "vendor_360_summary.csv", index=False)

cross_system_resolution_summary = (
    vendor_master.groupby(["golden_vendor_id"], as_index=False)
    .agg(
        source_record_count=("source_vendor_id", "count"),
        distinct_source_systems=("source_system", "nunique"),
        distinct_vendor_names=("vendor_name_raw", "nunique")
    )
    .sort_values(["distinct_source_systems", "source_record_count"], ascending=False)
)

cross_system_resolution_summary.to_csv(PROJECT_DIR / "cross_system_resolution_summary.csv", index=False)

top_duplicate_clusters = (
    vendor_master.groupby(["golden_vendor_id", "vendor_name_normalized"], as_index=False)
    .agg(
        source_record_count=("source_vendor_id", "count"),
        distinct_source_systems=("source_system", "nunique")
    )
    .sort_values(["source_record_count", "distinct_source_systems"], ascending=False)
    .head(25)
)

top_duplicate_clusters.to_csv(PROJECT_DIR / "top_duplicate_clusters.csv", index=False)

spend_before_resolution = (
    finance_transactions
    .merge(vendor_master[["source_system", "source_vendor_id", "vendor_name_raw"]], on=["source_system", "source_vendor_id"], how="left")
    .groupby("vendor_name_raw", as_index=False)
    .agg(total_spend_usd=("amount_usd", "sum"))
    .sort_values("total_spend_usd", ascending=False)
)

spend_after_resolution = (
    transactions_enriched.groupby("golden_vendor_name", as_index=False)
    .agg(total_spend_usd=("amount_usd", "sum"))
    .sort_values("total_spend_usd", ascending=False)
)

spend_before_resolution.to_csv(PROJECT_DIR / "spend_before_resolution.csv", index=False)
spend_after_resolution.to_csv(PROJECT_DIR / "spend_after_resolution.csv", index=False)

def search_vendor_entity(query, top_k=5):
    docs = golden_vendor_master.apply(
        lambda r: (
            f"golden vendor {r['golden_vendor_name']} "
            f"category {r['vendor_category']} "
            f"state {r['vendor_state']} "
            f"risk {r['risk_rating']} "
            f"systems {r['source_systems']} "
            f"source records {r['source_record_count']}"
        ),
        axis=1
    )

    search_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    search_matrix = search_vectorizer.fit_transform(docs)
    q = search_vectorizer.transform([query.upper()])
    sims = cosine_similarity(q, search_matrix)[0]
    idx = np.argsort(sims)[::-1][:top_k]
    result = golden_vendor_master.iloc[idx].copy()
    result["similarity_score"] = sims[idx]
    return result.reset_index(drop=True)

sample_search = search_vendor_entity("BluePeak Consult LLC high risk vendor", top_k=5)
sample_search.to_csv(PROJECT_DIR / "sample_vendor_search_results.csv", index=False)

plt.figure(figsize=(12, 5))
top_chart = vendor_360.head(10)
plt.bar(top_chart["golden_vendor_name"], top_chart["total_spend_usd"])
plt.xticks(rotation=45, ha="right")
plt.title("Top Golden Vendors by Total Spend")
plt.ylabel("Total Spend USD")
plt.tight_layout()
plt.show()

monthly_spend = (
    transactions_enriched.groupby("posting_month", as_index=False)
    .agg(total_spend_usd=("amount_usd", "sum"))
    .sort_values("posting_month")
)

plt.figure(figsize=(12, 5))
plt.plot(monthly_spend["posting_month"], monthly_spend["total_spend_usd"], marker="o")
plt.xticks(rotation=45, ha="right")
plt.title("Monthly Spend After Vendor Resolution")
plt.ylabel("Total Spend USD")
plt.tight_layout()
plt.show()

print("\nVendor match pairs sample")
display(match_pairs_df.head(20))

print("\nGolden vendor master")
display(golden_vendor_master.head(20))

print("\nVendor 360 summary")
display(vendor_360.head(20))

print("\nSample vendor entity search")
display(sample_search)

print("\nGenerated files:")
for p in sorted(PROJECT_DIR.glob("*")):
    print("-", p.name)

Project directory: /content/enterprise_finance_master_data_resolution_output
Raw vendor master shape: (460, 12)


,source_system,source_vendor_id,vendor_name_raw,tax_id,vendor_category,vendor_state,address_line_1,city,payment_terms_days,risk_rating,active_flag,canonical_seed_name
0,SAP_FI,SAP_00001,Prime Utility Parts,601903562,Consulting,NY,350 Market St,Hartford,60,Low,1,Prime Utility Parts
1,SAP_FI,SAP_00002,GreenVolt Systems LLC,112612231,Consulting,MA,716 Main St,Albany,30,Medium,1,GreenVolt Systems
2,SAP_FI,SAP_00003,alpha grid services,321478759,Electrical Components,NY,814 Industrial Rd,Newark,15,Medium,1,Alpha Grid Services
3,SAP_FI,SAP_00004,Helios Analytics Inc,757289253,Logistics,NY,926 Main St,Providence,45,High,1,Helios Analytics
4,SAP_FI,SAP_00005,EverField Maintenance LLC,326992101,IT Services,RI,146 Lake Dr,Hartford,15,Medium,1,EverField Maintenance


Golden vendor master shape: (21, 10)


,golden_vendor_id,golden_vendor_name,golden_tax_id,vendor_category,vendor_state,payment_terms_days,risk_rating,active_flag,source_record_count,source_systems
0,GV00001,PRIME UTILITYITY PARTS,601903562,Consulting,NJ,60,Low,1,31,"ERP_STAGE, SAP_AP, SAP_FI"
1,GV00002,GREENVOLT SYSTEMSTEMS,112612231,Maintenance,CT,30,Medium,1,27,"ERP_STAGE, SAP_AP, SAP_FI"
2,GV00003,ALPHA GRID SERVICES,321478759,Logistics,ME,45,Low,1,17,"ERP_STAGE, SAP_AP, SAP_FI"
3,GV00004,HELIOS ANALYTICS,757289253,Logistics,RI,60,Low,1,17,"ERP_STAGE, SAP_AP, SAP_FI"
4,GV00005,EVERFIELD MAINTENANCEENANCE,326992101,Consulting,NY,15,High,1,22,"ERP_STAGE, SAP_AP, SAP_FI"


KeyError: 'vendor_category'